In [ ]:
# Environment setup
#
# The kernel for this notebook runs a uv-managed virtualenv, which does not
# include pip (`uv venv` skips it). That is why the book's original `%pip ...`
# cell failed with "No module named pip" without installing anything.
#
# This venv also already runs the langchain 1.x line. The book's 0.3.x pins
# would downgrade it, so install only what this chapter needs, using uv against
# this kernel's own interpreter. Re-running this cell is safe.
#
# langchain-core is pinned to the version already installed so the resolver does
# not pull a newer core into the venv that every chapter shares.
import sys

# This chapter needs nothing beyond the shared stack.

# Verify what this notebook depends on
import importlib.metadata as md

for pkg in [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-openai",
    "langchain-classic",
    "langchain-text-splitters",
    "chromadb",
    "rank-bm25",
    "PyPDF2",
    "python-dotenv",
    "pydantic",
]:
    try:
        print(f"{pkg:28} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:28} MISSING")

# No kernel restart is needed unless this cell actually installed something new.

In [ ]:
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
class hub:
    # `from langchain import hub` was removed in langchain 1.x; pull prompts
    # through the LangSmith client instead (works without an API key for
    # public prompts such as jclemens24/rag-prompt).
    _client = Client()
    @staticmethod
    def pull(name):
        return hub._client.pull_prompt(name)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv, find_dotenv
from langchain_core.prompts import PromptTemplate
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents.base import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI

# variables
_ = load_dotenv(dotenv_path='env.txt')
# other variables
os.environ['AZURE_OPENAI_API_KEY'] = os.getenv('AZURE_OPENAI_API_KEY')
os.environ['AZURE_OPENAI_ENDPOINT'] = os.getenv('AZURE_OPENAI_ENDPOINT')
os.environ['AZURE_OPENAI_API_VERSION'] = os.getenv('AZURE_OPENAI_API_VERSION')

# NOTE: these are the deployment names that actually exist on this Azure OpenAI
# resource. "gpt-4o" and "text-embedding-3-large" are NOT deployed and return
# 404 DeploymentNotFound.
embedding_function = AzureOpenAIEmbeddings(
    api_key=os.environ['AZURE_OPENAI_API_KEY'],
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_version=os.environ['AZURE_OPENAI_API_VERSION'],
    model="text-embedding-3-small",
    # S0 tier throttles hard on embedding-heavy cells (e.g. SemanticChunker).
    # chunk_size caps how many texts go in one request: the 2048 default sends
    # ~1,900 sentences (~90k tokens) in a single call, which blows the tier's
    # rate limit on every attempt, so retrying cannot help. Small batches plus
    # backoff let the work through.
    chunk_size=16,
    max_retries=10,
)
llm = AzureChatOpenAI(
    azure_deployment="gpt-4o-mini",
    api_version=os.environ['AZURE_OPENAI_API_VERSION'],
    temperature=0,
    max_retries=10)
pdf_path = "google-2023-environmental-report.pdf"
collection_name = "google_environmental_report"
str_output_parser = StrOutputParser()
user_query = "What are Google's environmental initiatives?"

In [23]:
#### INDEXING ####

In [24]:
# PDF Loader
docs = []
with open(pdf_path, "rb") as pdf_file:
    pdf_reader = PdfReader(pdf_file)
    pdf_text = "".join(page.extract_text() for page in pdf_reader.pages)
    docs = [Document(page_content=page) for page in pdf_text.split("\n\n")]

In [25]:
# RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=200
)

splits = recursive_splitter.split_documents(docs)

In [ ]:
dense_documents = [Document(page_content=doc.page_content, metadata={"id": str(i), "search_source": "dense"}) for i, doc in enumerate(splits)]
sparse_documents = [Document(page_content=doc.page_content, metadata={"id": str(i), "search_source": "sparse"}) for i, doc in enumerate(splits)]

In [26]:
# Chroma Vector Store
chroma_client = chromadb.Client()
vectorstore = Chroma.from_documents(
    documents=dense_documents,
    embedding=embedding_function,
    collection_name=collection_name,
    client=chroma_client
)

In [27]:
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
sparse_retriever = BM25Retriever.from_documents(sparse_documents, k=10)
ensemble_retriever = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5], c=0, k=10)

In [28]:
#### RETRIEVAL and GENERATION ####

In [29]:
# Prompt - ignore LangSmith warning, you will not need langsmith for this coding exercise
prompt = hub.pull("jclemens24/rag-prompt")

In [30]:
# Relevance check prompt
relevance_prompt_template = PromptTemplate.from_template(
    """
    Given the following question and retrieved context, determine if the context is relevant to the question.
    Provide a score from 1 to 5, where 1 is not at all relevant and 5 is highly relevant.
    Return ONLY the numeric score, without any additional text or explanation.

    Question: {question}
    Retrieved Context: {retrieved_context}

    Relevance Score:"""
)

In [31]:
#### OUTPUT PARSERS ####

In [32]:
from langchain_core.output_parsers import StrOutputParser
str_output_parser = StrOutputParser()

In [ ]:
# JSON Output Parser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.outputs import Generation
import json

# Define FinalOutputModel for JSON output
class FinalOutputModel(BaseModel):
    relevance_score: float = Field(description="The relevance score of the retrieved context to the question")
    answer: str = Field(description="The final answer to the question")

# JsonOutputParser's argument is `pydantic_object`; the book passes
# `pydantic_model`, which pydantic accepts silently and then ignores,
# leaving the parser with no schema to validate against.
json_parser = JsonOutputParser(pydantic_object=FinalOutputModel)

In [34]:
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
def extract_score(llm_output):
    try:
        score = float(llm_output.strip())
        return score
    except ValueError:
        return 0

def format_json_output(x):
    json_output = {
        "relevance_score": extract_score(x['relevance_score']),
        "answer": x['answer'],
    }
    return json_parser.parse_result([Generation(text=json.dumps(json_output))])

def conditional_answer(x):
    relevance_score = extract_score(x['relevance_score'])
    if relevance_score < 4:
        return "I don't know."
    else:
        return format_json_output(x)

In [37]:
rag_chain = (
    RunnableParallel({"context": ensemble_retriever, "question": RunnablePassthrough()})
    | RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | RunnableParallel(
        {
            "relevance_score": (
                RunnablePassthrough()
                | (
                    lambda x: relevance_prompt_template.format(
                        question=x["question"], retrieved_context=x["context"]
                    )
                )
                | llm
                | str_output_parser
            ),
            "answer": (
                RunnablePassthrough()
                | prompt
                | llm
                | str_output_parser
            ),
        }
    )
    | RunnablePassthrough().assign(final_result=conditional_answer)
)

In [38]:
result = rag_chain.invoke(user_query)

print(f"Original Question: {user_query}\n")
print(f"Relevance Score: {result['relevance_score']}\n")
print(f"Final Answer:\n{result['final_result']['answer']}\n\n")
print(f"Final JSON Output:\n{result}\n\n")

Original Question: What are Google's environmental initiatives?

Relevance Score: 5

Final Answer:
Google's environmental initiatives include empowering individuals to take action through sustainability features in products like Google Maps, Google Nest thermostats, and Google Flights. They aim to help individuals, cities, and partners collectively reduce 1 gigaton of carbon equivalent emissions annually by 2030. Google also works with suppliers to reduce energy consumption and greenhouse gas emissions, as well as engages in public policy advocacy for low-carbon economies. Additionally, Google is involved in initiatives like the iMasons Climate Accord, ReFED, and supporting projects with The Nature Conservancy to address environmental challenges. They also focus on operating sustainably in their own operations, such as promoting sustainable consumption of public goods and engaging with coalitions like the RE-Source Platform. Google is also working on renewable energy solutions and usin